import torch
import numpy as np
import torch.nn.functional as F

R_nab = torch.rand(16, 4, 4)
I, J = torch.triu_indices(len(R_nab), len(R_nab), 1)

R_nnabb = R_nab[I, :, :, None].maximum(R_nab[J, :, None])
R_nnaab = R_nab[I, :, None, :].maximum(R_nab[J, None, :])
X_nab = torch.rand_like(R_nab, requires_grad=True)
W_ab_nab = torch.randn((len(X_nab), X_nab.shape[1] * X_nab.shape[2]), requires_grad=True)
opt = torch.optim.Adam([X_nab, W_ab_nab], 0.01)
for i, (k_sharp, k_far) in enumerate(zip(torch.logspace(0.5, 1.5, 3000), torch.logspace(0, 0.5, 3000)), 1):
    opt.zero_grad()
    logp_ab_nab = W_ab_nab.log_softmax(-1).unflatten(-1, (X_nab.shape[1], -1))
    logp_aclose_nnabb = F.logsigmoid((R_nnabb - abs(X_nab[I, :, :, None] - X_nab[J, :, None, :])) * k_sharp)
    logp_abb_nnabb = logp_ab_nab[I, :, :, None] + logp_ab_nab[J, :, None, :]
    logp_aallfar = (-(logp_aclose_nnabb + logp_abb_nnabb).expm1()).flatten(-3).prod(-1).log().mean()
    logp_bclose_nnabb = F.logsigmoid((R_nnaab - abs(X_nab[I, :, None, :] - X_nab[J, None, :, :])) * k_sharp)
    logp_aab_nnabb = logp_ab_nab[I, :, None, :] + logp_ab_nab[J, None, :, :]
    logp_ballfar = (-(logp_bclose_nnabb + logp_aab_nnabb).expm1()).flatten(-3).prod(-1).log().mean()
    logp_allfar = logp_aallfar + logp_ballfar
    los_allfar = -logp_allfar * k_far
    los_d = d = X_nab.std()
    (los_allfar + los_d).backward()
    opt.step()
    if not i % 100:
        print(logp_allfar.data.exp(), d.data, los_allfar.data, los_d.data)

from matplotlib.cm import rainbow
import matplotlib.pyplot as plt
def getlos(R_nab, Pabt_nabt, X_nabt, k_softmax=1, k_repulse=1, k_entropy=1, k_overlap=1, k_span=1, hard=False):
    pabt_nabtflat = Pabt_nabt.flatten(1).softmax(1)
    # pabt_nabtflat = F.gumbel_softmax(Pabt_nabt.flatten(1), k_softmax, hard)
    pabt_nabt = pabt_nabtflat.reshape(X_nabt.shape)
    # adiff_Nabbtt = X_nabt[I, :, :, None, :, None] - X_nabt[J, :, None, :, None, :]
    # overlapa_Nabbtt = (R_nab[I, :, :, None, None, None] + adiff_Nabbtt).maximum(R_nab[J, :, None, :, None, None] - adiff_Nabbtt)
    overlapa_Nabbtt = (X_nabt[I, :, :, None, :, None] + R_nab[I, :, :, None, None, None]).minimum(X_nabt[J, :, None, :, None, :] + R_nab[J, :, None, :, None, None]) - X_nabt[I, :, :, None, :, None].maximum(X_nabt[J, :, None, :, None, :])
    overlapa_Nabbtt = overlapa_Nabbtt.relu()
    # overlapa_Nabbtt = F.softplus(overlapa_Nabbtt, k_repulse)
    pabbtt_Nabbtt = pabt_nabt[I, :, :, None, :, None] * pabt_nabt[J, :, None, :, None, :]
    overlapa = (overlapa_Nabbtt.flatten(1)[:, None] @ pabbtt_Nabbtt.flatten(1)[:, :, None]).mean()
    # bdiff_Nabbtt = X_nabt[I, :, None, :, :, None] - X_nabt[J, None, :, :, None, :]
    # overlapb_Naabtt = (R_nab[I, :, None, :, None, None] + bdiff_Nabbtt).minimum(R_nab[J, None, :, :, None, None] - bdiff_Nabbtt)
    overlapb_Naabtt = (X_nabt[I, :, None, :, :, None] + R_nab[I, :, None, :, None, None]).minimum(X_nabt[J, None, :, :, None, :] + R_nab[J, None, :, :, None, None]) - X_nabt[I, :, None, :, :, None].maximum(X_nabt[J, None, :, :, None, :])
    overlapb_Naabtt = overlapb_Naabtt.relu()
    # overlapb_Naabtt = F.softplus(overlapb_Naabtt, k_repulse)
    paabtt_Naabtt = pabt_nabt[I, :, None, :, :, None] * pabt_nabt[J, None, :, :, None, :]
    overlapb = (overlapb_Naabtt.flatten(1)[:, None] @ paabtt_Naabtt.flatten(1)[:, :, None]).mean()
    min_n = (X_nabt.flatten(1)[:, None] @ pabt_nabtflat[:, :, None]).squeeze()
    max_n = min_n + (R_nab.flatten(1)[:, None] @ pabt_nabt.sum(3).flatten(1)[:, :, None]).squeeze()
    spansoft = (max_n * k_span).logsumexp(0) / k_span - (min_n * -k_span).logsumexp(0) / -k_span
    # with torch.no_grad():
    #     spanhard = max_n.max() - min_n.min()
    #     isolid_n = pabt_nabtflat.argmax(1)
    #     asolid_n, bsolid_n, tsolid_n = torch.unravel_index(isolid_n, X_nabt.shape[1:])
    #     minsolid_n = X_nabt[torch.arange(len(R_nab)), asolid_n, bsolid_n, tsolid_n]
    #     maxsolid_n = minsolid_n + R_nab[torch.arange(len(R_nab)), asolid_n, bsolid_n]
    #     overlapsolid_N = (maxsolid_n[I] - minsolid_n[J]).minimum(maxsolid_n[J] - minsolid_n[I]).relu()
    #     overlapasolid = overlapsolid_N @ (asolid_n[I] == asolid_n[J]).float() / len(I)
    #     overlapbsolid = overlapsolid_N @ (bsolid_n[I] == bsolid_n[J]).float() / len(I)
    #     spansolid = maxsolid_n.max() - minsolid_n.min()
    pab_nabflat = pabt_nabt.sum(3).flatten(1)
    entropy = -(pab_nabflat[:, None] @ (pab_nabflat + 1e-9).log()[:, :, None]).mean()
    return entropy * k_entropy + (overlapa + overlapb) * k_overlap + spansoft, {
        "entropy": entropy.data,
        "overlapa": overlapa.data,
        # "overlapasolid": overlapasolid,
        "overlapb": overlapb.data,
        # "overlapbsolid": overlapbsolid,
        "spansoft": spansoft.data,
        # "spanhard": spanhard,
        # "spansolid": spansolid,
    }
def draw(R_nab, Pabt_nabt, X_nabt):
    a, b, t = torch.unravel_index(Pabt_nabt.flatten(1).argmax(1), X_nabt.shape[1:])
    x = X_nabt.data[torch.arange(len(a)), a, b, t]
    y = x + R_nab[torch.arange(len(a)), a, b]
    cvs = torch.zeros((R_nab.shape[1] + R_nab.shape[2], 20, 3)).numpy()
    for a, b, x, y, c in zip(a, b, (cvs.shape[1] / (y.max() - x.min()) * (x - x.min())).long(), (cvs.shape[1] / (y.max() - x.min()) * (y - x.min())).long(), rainbow(np.linspace(0, 1, len(a)))[:, :3] / 2):
        cvs[a, x:y] /= 2
        cvs[a, x:y] += c
        cvs[R_nab.shape[1] + b, x:y] /= 2
        cvs[R_nab.shape[1] + b, x:y] += c
    plt.imshow((cvs * 255.9).astype("B"))
    plt.show()
trial = 4
X_nabt = torch.rand(*R_nab.shape, trial, requires_grad=True)
Pabt_nabt = torch.randn(*R_nab.shape, trial, requires_grad=True)
opt = torch.optim.AdamW(
    [
        {"params": X_nabt, "lr": 1e-3},
        {"params": Pabt_nabt, "lr": 1e-3},
    ],
    weight_decay=1e-5,
)
for i, arg in enumerate(
    dict(zip("k_softmax k_repulse k_entropy k_overlap k_span".split(), i))
    for i in zip(
        torch.logspace(np.log10(1), np.log10(0.5), 5000),
        torch.logspace(1, 1, 5000),
        torch.logspace(-1, 0.5, 5000),
        torch.logspace(0, 0.5, 5000),
        torch.logspace(0, 2, 5000),
    )
):
    opt.zero_grad()
    los, inf = getlos(R_nab, Pabt_nabt, X_nabt, **arg)
    los.backward()
    torch.nn.utils.clip_grad_norm_([X_nabt, Pabt_nabt], 1)
    opt.step()
    if not (i + 1) % 100:
        print(los.data, inf)
        draw(R_nab, Pabt_nabt, X_nabt)

def getlos(R_nab, Ptab_ntab, X_ntab, k_softmax=None, k_softplus=None, k_span=1, w_entropy=1, w_overlap=1, w_spansoft=1, w_span=0, hard=False):
    Ptab_ntab = Ptab_ntab.flatten(1)
    if k_softmax or hard:
        Ptab_ntab = F.gumbel_softmax(Ptab_ntab, k_softmax or 1, hard)
    else:
        Ptab_ntab = Ptab_ntab.softmax(1)
    Ptab_ntab = Ptab_ntab.reshape(X_ntab.shape)

    overlapa_Nttabb = X_ntab[I, :, None, :, :, None] - X_ntab[J, None, :, :, None, :]
    overlapa_Nttabb = (R_nab[I, None, None, :, :, None] + overlapa_Nttabb).minimum(R_nab[J, None, None, :, None, :] - overlapa_Nttabb)
    if k_softplus:
        overlapa_Nttabb = F.softplus(overlapa_Nttabb, k_softplus)
    else:
        overlapa_Nttabb = overlapa_Nttabb.relu()
    pttabb_Nttabb = Ptab_ntab[I, :, None, :, :, None] * Ptab_ntab[J, None, :, :, None, :]
    overlap = (overlapa_Nttabb.flatten(1)[:, None] @ pttabb_Nttabb.flatten(1)[:, :, None]).sum()

    overlapb_Nttaab = X_ntab[I, :, None, :, None, :] - X_ntab[J, None, :, None, :, :]
    overlapb_Nttaab = (R_nab[I, None, None, :, None, :] + overlapb_Nttaab).minimum(R_nab[J, None, None, None, :, :] - overlapb_Nttaab)
    if k_softplus:
        overlapb_Nttaab = F.softplus(overlapb_Nttaab, k_softplus)
    else:
        overlapb_Nttaab = overlapb_Nttaab.relu()
    pttaab_Nttaab = Ptab_ntab[I, :, None, :, None, :] * Ptab_ntab[J, None, :, None, :, :]
    overlap = overlap + (overlapb_Nttaab.flatten(1)[:, None] @ pttaab_Nttaab.flatten(1)[:, :, None]).sum()

    min_n = (X_ntab.broadcast_to(Ptab_ntab.shape).flatten(1)[:, None, :] * Ptab_ntab.flatten(1)[:, :, None])[:, 0, 0]
    max_n = ((X_ntab + R_nab[:, None]).flatten(1)[:, None, :] @ Ptab_ntab.sum(1).flatten(1)[:, :, None])[:, 0, 0]
    spansoft = (max_n * k_span).logsumexp(0) / k_span - (min_n * -k_span).logsumexp(0) / -k_span
    span = max_n.max() - min_n.min()

    entropy = -(Ptab_ntab.flatten(1)[:, None] @ (Ptab_ntab.flatten(1) + 1e-9).log()[:, :, None]).mean()
    return w_entropy * entropy + w_overlap * overlap + w_spansoft * spansoft + w_span * span, {
        "entropy": entropy.data,
        "overlap": overlap.data,
        "spansoft": spansoft.data,
        "span": span.data,
    }


def draw(R_nab, Ptab_ntab, X_ntab):
    t, a, b = torch.unravel_index(Ptab_ntab.flatten(1).argmax(1), Ptab_ntab.shape[1:])
    x = X_ntab.broadcast_to(Ptab_ntab.shape)[torch.arange(len(a)), t, a, b]
    y = x + R_nab[torch.arange(len(a)), a, b]
    mi = x.min()
    ma = y.max()
    cvs = torch.ones((R_nab.shape[1] + R_nab.shape[2] + 1, 50, 3))
    for a, b, x, y, c in zip(
        a,
        b,
        (cvs.shape[1] * (x - mi) / (ma - mi)).long(),
        (cvs.shape[1] * (y - mi) / (ma - mi)).long(),
        torch.tensor(plt.cm.rainbow(torch.linspace(0, 1, len(a)))[:, :3], dtype=torch.float) / 2,
    ):
        cvs[a, x:y] /= 2
        cvs[a, x:y] += c
        cvs[R_nab.shape[1] + b, x:y] /= 2
        cvs[R_nab.shape[1] + b, x:y] += c
    plt.imshow((cvs * 255).to(torch.uint8))
    plt.show()

X_ntab = torch.rand(len(R_nab), 1, *R_nab.shape[1:], requires_grad=True)
Ptab_ntab = torch.randn(*X_ntab.shape, requires_grad=True)
opt = torch.optim.Adam([X_ntab, Ptab_ntab], 1e-3)
for i, arg in enumerate(
    dict(zip("k_span w_entropy w_overlap".split(), i))
    for i in zip(
        torch.logspace(0, 0, 5000),
        torch.logspace(0, 1, 5000),
        torch.logspace(0, 0, 5000),
    )
):
    opt.zero_grad()
    los, inf = getlos(R_nab, Ptab_ntab, X_ntab, **arg, w_spansoft=0, w_span=1)
    los.backward()
    torch.nn.utils.clip_grad_norm_([X_ntab, Ptab_ntab], 1)
    opt.step()
    if not (i + 1) % 100:
        clear_output(True)
        with torch.no_grad():
            draw(R_nab, Ptab_ntab, X_ntab)
            print(los.data)
            for i, j in inf.items():
                print(i, j)
            los, inf = getlos(R_nab, Ptab_ntab, X_ntab, **arg, hard=True)
            print(los.data)
            for i, j in inf.items():
                print(i, j)

In [ ]:
from ipywidgets import Output

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from ipywidgets import FloatLogSlider, Output


def mmlstax(a, b):
    return (a[..., None, :] @ b[..., :, None])[..., 0, 0]


def getlos(r_n, x_n, a_na, b_nb, ksoftmax=0, kspan=1, koverlap=0, wentropy=1, woverlap=1, wspansoft=1, wspan=0, hard=False):
    overlap_N = x_n[..., I] - x_n[..., J]
    overlap_N = (r_n[..., I] + overlap_N).minimum(r_n[..., J] - overlap_N)
    if koverlap > 1e-3:
        overlap_N = F.softplus(overlap_N, koverlap)
    else:
        overlap_N = overlap_N.relu()
    if ksoftmax > 1e-3:
        a_na = F.gumbel_softmax(a_na, ksoftmax or 1, hard)
        b_nb = F.gumbel_softmax(b_nb, ksoftmax or 1, hard)
    else:
        a_na = a_na.softmax(-1)
        b_nb = b_nb.softmax(-1)
    entropy = -mmlstax(a_na.flatten(-2), (a_na.flatten(-2) + 1e-9).log()) - mmlstax(b_nb.flatten(-2), (b_nb.flatten(-2) + 1e-9).log())
    overlap = mmlstax(mmlstax(a_na[..., I, :], a_na[..., J, :]), overlap_N) + mmlstax(mmlstax(b_nb[..., I, :], b_nb[..., J, :]), overlap_N)
    span = x_n.amax(-1) - x_n.amin(-1)
    if kspan > 1e-3:
        spansoft = 1 / kspan * (x_n * kspan).logsumexp(-1) - 1 / -kspan * (x_n * -kspan).logsumexp(-1)
    else:
        spansoft = span
    los = 1 / (wentropy + woverlap + wspansoft + wspan) * (wentropy * entropy + woverlap * overlap + wspansoft * spansoft + wspan * span)
    return los, {
        "entropy": entropy.data,
        "overlap": overlap.data,
        "spansoft": spansoft.data,
        "span": span.data,
    }


def draw(r_n, x_n, a_na, b_nb, los):
    los = torch.unravel_index(los.argmin(), los.shape)
    x_n = x_n[los]
    y_n = x_n + r_n[los]
    mi = x_n.min()
    ma = y_n.max()
    cvs = torch.zeros((a_na.shape[-1] + 1 + b_nb.shape[-1], (a_na.shape[-1] + 1 + b_nb.shape[-1]) * 4, 3))
    for a, b, x, y, c in zip(
        a_na[los].argmax(1),
        a_na.shape[-1] + 1 + b_nb[los].argmax(1),
        (cvs.shape[1] * (x_n - mi) / (ma - mi)).long(),
        (cvs.shape[1] * (y_n - mi) / (ma - mi)).long(),
        torch.tensor(plt.cm.rainbow(torch.linspace(0, 1, len(x_n)))[:, :3], dtype=torch.float) / 2,
    ):
        cvs[a, x:y] += c
        cvs[b, x:y] += c
    plt.figure(figsize=(24, 12))
    plt.imshow((cvs.clip(max=1) * 255).to(torch.uint8))
    plt.show()


kspan = FloatLogSlider(min=-3, max=3, description="kspan")
wentropy = FloatLogSlider(min=-3, max=3, description="wentropy")
woverlap = FloatLogSlider(4, min=-3, max=3, description="woverlap")
wspansoft = FloatLogSlider(min=-3, max=3, description="wspansoft")
wspan = FloatLogSlider(1e-3, min=-3, max=3, description="wspan")
out = Output()
display(kspan, wentropy, woverlap, wspansoft, wspan, out)
r_n = torch.rand(256) + 0.5
I, J = torch.triu_indices(len(r_n), len(r_n), 1)
x_n = torch.randn(4, len(r_n), requires_grad=True)
a_na = torch.randn(*x_n.shape, 8, requires_grad=True)
b_nb = torch.randn(*x_n.shape, 16, requires_grad=True)
opt = torch.optim.Adam((x_n, a_na, b_nb), 0.01)
for i in range(99999999):
    opt.zero_grad()
    arg = dict(kspan=kspan.value, wentropy=wentropy.value, woverlap=woverlap.value, wspansoft=wspansoft.value, wspan=wspan.value)
    los, inf = getlos(r_n, x_n, a_na, b_nb, **arg)
    los.mean().backward()
    torch.nn.utils.clip_grad_norm_((x_n, a_na, b_nb), 1)
    opt.step()
    if not (i + 1) % 100:
        with torch.no_grad(), out:
            out.clear_output(True)
            draw(r_n, x_n, a_na, b_nb, los)
            print(arg)
            print(los)
            for i, j in inf.items():
                print(i, j)
            los, inf = getlos(r_n, x_n, a_na, b_nb, **arg, hard=True)
            print(los)
            for i, j in inf.items():
                print(i, j)

In [ ]:
kspanslide.value